# CryptoHypothesisLab — 3/3: Pre-Move Early Warning (offline)

**Otvoriti u potpuno novom Google Colab runtimeu.** Nema montiranja Drivea. Učitati **samo** ranije verificirani privatni Development ZIP preko browsera. Ne šalji ZIP, plan ili izvještaj na javni GitHub.

Test mjeri sve alarme, uključujući **lažne alarme**, na svakoj zatvorenoj minuti. Asia High/Low su samo **ekstremi C1 minutnih zatvaranja** (00:00–07:59 UTC), a C2 delta su **izvršene agresivne trgovine** — ne L2 OFI. Svi rezultati su `DESCRIPTIVE_DEVELOPMENT_ONLY`, bez nezavisne potvrde edgea. Vizualizacija epizoda koristi **ex-post** referentno vrijeme, ne pravi početak pomjeranja.


In [ ]:
# 1. Sigurnosna provjera i javni kod, bez montiranja Google Drivea.
import os,sys,subprocess,hashlib,json
from pathlib import Path
if any(Path(p).exists() for p in ('/content/drive/MyDrive','/content/gdrive/MyDrive','/gdrive/MyDrive')):
    raise RuntimeError('GOOGLE DRIVE JE MONTIRAN — izbriši runtime i otvori novu sesiju')
if any(os.environ.get(k) for k in ('GOOGLE_APPLICATION_CREDENTIALS','GOOGLE_OAUTH_ACCESS_TOKEN','GOOGLE_DRIVE_TOKEN')):
    raise RuntimeError('Google token/credential u okruženju')
repo=Path('/content/hypolab-premove-code')
if repo.exists():raise RuntimeError('Koristi potpuno novi runtime')
subprocess.run(['git','clone','--depth','1','https://github.com/DuStER1921/CryptoHypothesisLab.git',str(repo)],check=True,stdout=subprocess.DEVNULL)
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
sys.path.insert(0,str(repo/'src'))
import numpy as np
from hypolab.premove_warning import make_premove_plan,run_premove_study
from google.colab import files
print('Analyst PASS, bez Drive mounta. Git commit:',commit[:12])


In [ ]:
# 2. Učitaj samo privatni Development ZIP i nezavisno provjeri SHA.
uploaded=files.upload()
if len(uploaded)!=1:raise RuntimeError('Učitaj tačno jedan ZIP')
filename,payload=next(iter(uploaded.items()))
if not filename.startswith('HYPOLAB_DEV_') or not filename.endswith('.zip'):
    raise RuntimeError('Pogrešan tip ZIP paketa')
expected_hash=input('SHA-256 ZIP-a iz Windows PowerShell provjere (64 znaka): ').strip().lower()
if len(expected_hash)!=64 or any(c not in '0123456789abcdef' for c in expected_hash):
    raise RuntimeError('Neispravan očekivani SHA-256')
actual_hash=hashlib.sha256(payload).hexdigest()
if actual_hash!=expected_hash:raise RuntimeError('ZIP HASH MISMATCH — NE NASTAVLJAJ')
print('ZIP SHA256 PASS:',actual_hash)


In [ ]:
# 3. Registruj/presnimi kompletan fiksni plan PRIJE računanja ishoda.
plan=make_premove_plan(payload)
plan_path=Path('/content/HYPOLAB_PREMOVE_FIXED_PLAN.json')
if plan_path.exists():raise RuntimeError('Plan postoji — potreban je novi runtime')
plan_path.write_bytes(plan)
files.download(str(plan_path))
print('Plan sačuvan:',hashlib.sha256(plan).hexdigest())
print('3 mehanizma × 2 praga × 3 horizonta = 18 poređenja')


In [ ]:
# 4. Izvrši kompletan rani-detektor i ispiši sve promašaje.
report=run_premove_study(payload,plan)
print('INFERENCE:',report['inference'])
print('Minuta:',report['feature_minutes'],'C2 nepoznato:',report['c2_unknown_minutes'])
print('BOOK DEPTH:',report['book_depth'])
print('Rule | bps | h | alerti | tačan smjer | pogrešan | bez pokreta | lažni alarmi (%)')
for x in report['evaluations']:
    pct=None if x['false_alarm_fraction'] is None else round(x['false_alarm_fraction']*100,1)
    print(x['rule'],x['threshold_bps'],x['horizon_minutes'],x['evaluated_alerts'],
          x['correct_direction_hit'],x['wrong_direction_first_hit'],x['no_threshold_hit'],pct)
print('Ex-post burst proxy epizoda:',report['ex_post_burst_gallery']['total_proxy_episodes'])
print('Rani alarm / propusteni EX-POST proxy dogadjaji (nije prava onset detekcija):')
for rule,x in report['ex_post_burst_gallery']['event_proxy_warning_coverage'].items():
    print(rule,'pre-proxy upozorenja:',x['direction_matched_proxy_episodes'],
          'bez upozorenja:',x['proxy_episodes_missed'],
          'medijan lead min:',x['median_warning_minutes_before_proxy_anchor'])


In [ ]:
# 5. Vizualizacija ŠTA SE DOGAĐALO PRIJE ex-post reference pokreta.
# Ovo nije dokaz da je algoritam znao budućnost: epizode su izabrane nakon uvida u ishode.
import matplotlib.pyplot as plt
examples=report['ex_post_burst_gallery']['gallery']
if not examples:
    print('Nema dovoljno epizoda sa punim 30m prethodnog konteksta.')
else:
    index=0  # Odaberi 0..len(examples)-1 pa ponovno pokreni SAMO ovu ćeliju.
    if not 0<=index<len(examples):raise ValueError('Nevažeći index')
    ev=examples[index]
    w=ev['pre_reference_30min']+ev['event_and_post_30min']
    x=[r['relative_minute'] for r in w]
    print('EX-POST referenca:',ev['proxy_start_utc'],'smjer:',ev['direction'],
          '15m povrat bps:',ev['15m_signed_close_return_bps'])
    print('Rani alarmi prije reference:',ev['early_alerts'])
    fig,axes=plt.subplots(4,1,figsize=(11,11),sharex=True)
    axes[0].plot(x,[r['taker_imbalance'] for r in w]);axes[0].axhline(0,linestyle='--')
    axes[0].set_ylabel('C2 taker imbalance')
    axes[1].plot(x,[r['relative_taker_volume'] for r in w]);axes[1].axhline(1,linestyle='--')
    axes[1].set_ylabel('Volumen / ranijih 60m')
    axes[2].plot(x,[r['delta_imbalance_vs_previous5'] for r in w])
    axes[2].set_ylabel('Promjena delta omjera')
    axes[3].plot(x,[r['close_to_asia_high_bps'] for r in w],label='Do Asia High (CLOSE proxy)')
    axes[3].plot(x,[r['close_to_asia_low_bps'] for r in w],label='Do Asia Low (CLOSE proxy)')
    axes[3].set_ylabel('CLOSE prema Asia (bps)');axes[3].set_xlabel('Minute prema ex-post referenci')
    axes[3].legend()
    for ax in axes:
        ax.axvline(0,linestyle=':');ax.axvline(15,linestyle=':')
    fig.suptitle('PRE-MOVE: prethodnih 30m / tokom 15m / narednih 15m — SAMO opisno')
    plt.tight_layout();plt.show()


In [ ]:
# 6. Sačuvaj SAV izvještaj privatno — uključujući sve negativne rezultate.
report_path=Path('/content/HYPOLAB_PREMOVE_COMPLETE_REPORT.json')
if report_path.exists():raise RuntimeError('Izvještaj već postoji — ne prepisuj')
raw=(json.dumps(report,sort_keys=True,indent=2,ensure_ascii=False,allow_nan=False)+'\n').encode('utf-8')
report_path.write_bytes(raw)
files.download(str(report_path))
print('Privatni report SHA256:',hashlib.sha256(raw).hexdigest())
print('Nemoj dodavati report/ZIP na javni GitHub; nemoj montirati Drive u ovom runtimeu.')
